# ASEAN V2.1 final ensemble audit (RC3)

This notebook does not retrain. It consumes the frozen five-seed forecasts, rebuilds E1–E4 on the common universe, selects one ensemble on validation only, refits calibration, and compares PTCST with risk-only MVO/CA under C0/C1/C2.

In [ ]:
# Cell 1 — mount Drive and clone current code
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
from pathlib import Path
import subprocess, sys, shutil, pandas as pd
REPO = Path('/content/kltn')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','https://github.com/maiphuowng205/kltn.git',str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-colab.txt')], check=True)
DRIVE = Path('/content/drive/MyDrive')
print('Commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())

In [ ]:
# Cell 2 — restore only the already-built V2 data and saved five-seed runs
V2_DRIVE = DRIVE/'kltn'/'asean_v2'
DRIVE_RUN = DRIVE/'kltn'/'asean_v2_development'/'pooled_ptcst'
V2_DATA = Path('/content/asean_v2')
RUN_ROOT = Path('/content/asean_v2_runs/pooled_ptcst')
if not (V2_DRIVE/'model_ready'/'weekly_features_targets_v2').exists(): raise FileNotFoundError(f'Missing V2 dataset: {V2_DRIVE}')
if not DRIVE_RUN.exists(): raise FileNotFoundError(f'Missing saved seed runs: {DRIVE_RUN}')
for path in [V2_DATA, RUN_ROOT]:
    if path.exists(): shutil.rmtree(path)
shutil.copytree(V2_DRIVE, V2_DATA)
RUN_ROOT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_RUN, RUN_ROOT)
ORIGINAL_RUN_ROOT = RUN_ROOT  # replace with an independent frozen copy when available
print('V2 data:', V2_DATA)
print('Seed runs:', RUN_ROOT)

In [ ]:
# Cell 3 — run reconciliation, E1–E4 selection, calibration, attribution and bootstrap
AUDIT_OUT = Path('/content/asean_v21_final_audit')
if AUDIT_OUT.exists(): shutil.rmtree(AUDIT_OUT)
command = [sys.executable,str(REPO/'scripts/run_asean_v21_final_audit.py'),'--data-root',str(V2_DATA),'--run-root',str(RUN_ROOT),'--original-run-root',str(ORIGINAL_RUN_ROOT),'--output-dir',str(AUDIT_OUT),'--risk-aversion','50','--turnover-cap','.40','--bootstrap-draws','2000','--bootstrap-block','5']
result = subprocess.run(command, text=True, capture_output=True)
print('RETURN CODE:', result.returncode)
print('--- STDOUT ---')
print(result.stdout)
print('--- STDERR ---')
print(result.stderr)
if result.returncode != 0: raise RuntimeError('Final audit failed; see STDERR above.')
print('Audit output:', AUDIT_OUT)

In [ ]:
# Cell 4 — inspect the final decision tables
display(pd.read_csv(AUDIT_OUT/'ensemble_validation_selection.csv'))
display(pd.read_csv(AUDIT_OUT/'incremental_value_summary.csv'))
display(pd.read_csv(AUDIT_OUT/'incremental_bootstrap_ci.csv'))
display(pd.read_csv(AUDIT_OUT/'development_ensemble_audit_summary.csv') if (AUDIT_OUT/'development_ensemble_audit_summary.csv').exists() else pd.DataFrame())

In [ ]:
# Cell 5 — save immutable RC3 audit package to Drive
DRIVE_OUT = DRIVE/'kltn'/'v2_1_final_audit'
if DRIVE_OUT.exists(): shutil.rmtree(DRIVE_OUT)
shutil.copytree(AUDIT_OUT, DRIVE_OUT)
print('Saved RC3 audit package to:', DRIVE_OUT)